# EDA — Home Credit Application Data

**Week 1: Setup & Data Understanding**

This notebook covers initial exploration of the `application_train.csv` dataset from Home Credit. Goals:
- Check target distribution (imbalance)
- Map missing values
- Look at initial correlation of numeric features with the target
- Investigate anomalies (`DAYS_EMPLOYED`)
- Run the cleaning pipeline (`src/data_cleaning.py`) and save the results


## 1. Load Data


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/application_train.csv')
df.shape

(307511, 122)

In [2]:
df.head()

## 2. Target Distribution (Imbalance Check)

This dataset is imbalanced: only ~8% of applications default (`TARGET`=1). This matters for model evaluation later — plain accuracy would be misleading, so AUC/KS/Gini should be used instead.


In [3]:
df['TARGET'].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

## 3. Data Types Overview


In [4]:
df.dtypes.value_counts()  # count of numeric vs categorical columns

float64    65
int64      41
object     16
Name: count, dtype: int64

## 4. Missing Value Map

Check which columns have missing values and how severe it is. Columns with >50% missing are candidates for dropping or need special treatment (e.g. `_MODE`/`_AVG`/`_MEDI` housing features).


In [5]:
missing = df.isnull().mean().sort_values(ascending=False)

# columns with the highest missing value rate (>~50%) -> drop candidates
missing[missing > 0].head(30)

COMMONAREA_MEDI             0.698723
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MEDI    0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_MODE       0.683550
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
FLOORSMIN_AVG               0.678486
FLOORSMIN_MODE              0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_MEDI            0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_AVG             0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_MODE               0.593767
LANDAREA_AVG                0.593767
BASEMENTAREA_MEDI           0.585160
BASEMENTAREA_AVG            0.585160
BASEMENTAREA_MODE           0.585160
EXT_SOURCE_1                0.563811
NONLIVINGAREA_MODE          0.551792
NONLIVINGAREA_AVG           0.551792
NONLIVINGAREA_MEDI          0.551792
E

Columns with a moderate amount of missing values (< 55%) — these are better suited for imputation rather than dropping, including `EXT_SOURCE_2/3`, `OCCUPATION_TYPE`, and several `AMT_REQ_CREDIT_BUREAU_*` columns.


In [6]:
missing[(missing > 0) & (missing < 0.55)]

ELEVATORS_MEDI                  0.532960
ELEVATORS_AVG                    0.532960
ELEVATORS_MODE                   0.532960
WALLSMATERIAL_MODE               0.508408
APARTMENTS_MEDI                  0.507497
APARTMENTS_AVG                   0.507497
APARTMENTS_MODE                  0.507497
ENTRANCES_MEDI                   0.503488
ENTRANCES_AVG                    0.503488
ENTRANCES_MODE                   0.503488
LIVINGAREA_AVG                   0.501933
LIVINGAREA_MODE                  0.501933
LIVINGAREA_MEDI                  0.501933
HOUSETYPE_MODE                   0.501761
FLOORSMAX_MODE                   0.497608
FLOORSMAX_MEDI                   0.497608
FLOORSMAX_AVG                    0.497608
YEARS_BEGINEXPLUATATION_MODE     0.487810
YEARS_BEGINEXPLUATATION_MEDI     0.487810
YEARS_BEGINEXPLUATATION_AVG      0.487810
TOTALAREA_MODE                   0.482685
EMERGENCYSTATE_MODE              0.473983
OCCUPATION_TYPE                  0.313455
EXT_SOURCE_3                     0.

### Why is `OWN_CAR_AGE` missing ~66% of the time?

First check whether it's genuinely missing or simply a logical consequence of `FLAG_OWN_CAR` = 'N' (doesn't own a car, so `OWN_CAR_AGE` being empty makes sense). If so, this isn't missing data that needs imputing — it's structural missingness.


In [7]:
df.groupby('FLAG_OWN_CAR')['OWN_CAR_AGE'].apply(lambda x: x.isnull().mean())

FLAG_OWN_CAR
N    1.000000
Y    0.000048
Name: OWN_CAR_AGE, dtype: float64

**Insight:** confirmed — 100% of `OWN_CAR_AGE` missing values occur for people where `FLAG_OWN_CAR` = 'N'. So this is structural missingness, not a data quality issue — it can be filled with 0 or a separate flag, not dropped.


## 5. EXT_SOURCE Features vs Target

`EXT_SOURCE_1/2/3` are often the most predictive features in this dataset (external scores, similar to a credit score from another source). Check the mean per target group.


In [8]:
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    print(col)
    print(df.groupby('TARGET')[col].mean())
    print('---')

EXT_SOURCE_1
TARGET
0    0.511461
1    0.386968
Name: EXT_SOURCE_1, dtype: float64
---
EXT_SOURCE_2
TARGET
0    0.523479
1    0.410935
Name: EXT_SOURCE_2, dtype: float64
---
EXT_SOURCE_3
TARGET
0    0.520969
1    0.390717
Name: EXT_SOURCE_3, dtype: float64
---


**Insight:** all three `EXT_SOURCE_*` features are consistently lower for the default group (`TARGET`=1). This aligns with the negative correlation seen in the next section — the higher the external score, the lower the likelihood of default.


## 6. Initial Correlation of Numeric Features with the Target


In [9]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['TARGET'].sort_values()
print(correlations.head(10))   # most negative (higher value -> less likely to default)
print(correlations.tail(11))   # most positive (higher value -> more likely to default) - 11 because TARGET itself is included

EXT_SOURCE_3                 -0.178919
EXT_SOURCE_2                 -0.160472
EXT_SOURCE_1                 -0.155317
DAYS_EMPLOYED                 -0.044932
FLOORSMAX_AVG                 -0.044003
FLOORSMAX_MEDI                -0.043768
FLOORSMAX_MODE                -0.043226
AMT_GOODS_PRICE                -0.039645
REGION_POPULATION_RELATIVE    -0.037227
ELEVATORS_AVG                  -0.034199
Name: TARGET, dtype: float64
DAYS_REGISTRATION              0.041975
FLAG_DOCUMENT_3                 0.044346
REG_CITY_NOT_LIVE_CITY          0.044395
FLAG_EMP_PHONE                  0.045982
REG_CITY_NOT_WORK_CITY          0.050994
DAYS_ID_PUBLISH                 0.051457
DAYS_LAST_PHONE_CHANGE          0.055218
REGION_RATING_CLIENT            0.058899
REGION_RATING_CLIENT_W_CITY     0.060893
DAYS_BIRTH                      0.078239
TARGET                          1.000000
Name: TARGET, dtype: float64


**Insight:** all linear correlations are weak (below 0.2), which is expected for credit risk data. `EXT_SOURCE_*` is the strongest (negative), followed by `DAYS_BIRTH` (positive — younger/larger value since `DAYS_BIRTH` is negative, means higher risk). This weak correlation is why a non-linear model (XGBoost) + feature engineering will be needed later, not just a linear model.


## 7. `DAYS_EMPLOYED` Anomaly

`DAYS_EMPLOYED` should be negative (days since starting work, counted backward from the application date). But there's a value of `365243` that is clearly a placeholder/error.


In [10]:
print(df['DAYS_EMPLOYED'].describe())
print((df['DAYS_EMPLOYED'] == 365243).sum())

count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max       365243.000000
Name: DAYS_EMPLOYED, dtype: float64
55374


**Fix:** create the `DAYS_EMPLOYED_ANOMALY` flag **before** replacing the value with NaN, so the anomaly information isn't lost (it can become its own feature).


In [11]:
# reload fresh
df = pd.read_csv('../data/raw/application_train.csv')

# 1. create the flag first BEFORE replacing
df['DAYS_EMPLOYED_ANOMALY'] = df['DAYS_EMPLOYED'] == 365243

# 2. now replace
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# 3. check
print(df['DAYS_EMPLOYED_ANOMALY'].value_counts())
print(df.groupby('DAYS_EMPLOYED_ANOMALY')['TARGET'].mean())

DAYS_EMPLOYED_ANOMALY
False    252137
True      55374
Name: count, dtype: int64
DAYS_EMPLOYED_ANOMALY
False    0.086600
True     0.053996
Name: TARGET, dtype: float64


**Insight:** the anomaly group (55,374 rows, ~18% of the data) actually has a lower default rate (5.4% vs 8.7%). This is most likely retirees (`NAME_INCOME_TYPE` = 'Pensioner'), not a random error — so this flag is worth keeping as a feature rather than just being dropped.


## 8. Simple Financial Ratios

Build 2 basic ratios to measure debt burden relative to income: total credit-to-income ratio, and monthly annuity-to-income ratio.


In [12]:
# 1. Look at the overall picture of the financial figures
print(df[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']].describe())

# 2. Compute total debt-to-income ratio & check the mean per TARGET group
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
print(df.groupby('TARGET')['CREDIT_INCOME_RATIO'].mean())

# 3. Compute monthly annuity-to-income ratio & check the mean per TARGET group
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
print(df.groupby('TARGET')['ANNUITY_INCOME_RATIO'].mean())

       AMT_INCOME_TOTAL    AMT_CREDIT    AMT_ANNUITY  AMT_GOODS_PRICE
count      3.075110e+05  3.075110e+05  307499.000000     3.072330e+05
mean       1.687979e+05  5.990260e+05   27108.573909     5.383962e+05
std        2.371231e+05  4.024908e+05   14493.737315     3.694465e+05
min        2.565000e+04  4.500000e+04    1615.500000     4.050000e+04
25%        1.125000e+05  2.700000e+05   16524.000000     2.385000e+05
50%        1.471500e+05  5.135310e+05   24903.000000     4.500000e+05
75%        2.025000e+05  8.086500e+05   34596.000000     6.795000e+05
max        1.170000e+08  4.050000e+06  258025.500000     4.050000e+06
TARGET
0    3.963729
1    3.887438
Name: CREDIT_INCOME_RATIO, dtype: float64
TARGET
0    0.180530
1    0.185482
Name: ANNUITY_INCOME_RATIO, dtype: float64


**Insight:** both ratios are nearly identical between the default and non-default groups — the difference is small, so they're not very predictive on their own. They likely need interaction with other features (e.g. `EXT_SOURCE`, occupation type) to be more useful, or a different formulation (e.g. annuity-to-credit ratio).


## 9. Run the Cleaning Pipeline (`src/data_cleaning.py`)

All the treatments above (anomaly flagging, handling structural missingness, etc.) have been consolidated into a single function `clean_application_data` in `src/data_cleaning.py`. Reload the raw data, run the pipeline, and confirm the result is clean (0 missing values).


In [13]:
import sys
sys.path.append('../src')
from data_cleaning import clean_application_data

df_raw = pd.read_csv('../data/raw/application_train.csv')
df_clean = clean_application_data(df_raw)

print(df_clean.shape)
print(df_clean.isnull().sum().sum())  # total missing values across all columns, should be 0

(307511, 79)
0


## 10. Save Clean Data


In [14]:
df_clean.to_csv('../data/processed/application_train_clean.csv', index=False)

---
### Summary & Next Steps
- Target imbalance ~8% default → AUC/KS/Gini needed, not accuracy.
- `OWN_CAR_AGE` missing = structural (doesn't own a car), not a data quality issue.
- `DAYS_EMPLOYED` = 365243 is an anomaly (likely retirees), already flagged as `DAYS_EMPLOYED_ANOMALY`.
- `EXT_SOURCE_1/2/3` are the most predictive features so far.
- Simple income ratios (`CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`) aren't very predictive on their own yet.
- Clean data has been saved to `data/processed/application_train_clean.csv`, ready to move on to **Week 2: Anomaly Detection Layer**.
